This notebook uses the Aposemat IoT-23 dataset.

*Sebastian Garcia, Agustin Parmisano, & Maria Jose Erquiaga. (2020). IoT-23: A labeled dataset with malicious and benign IoT network traffic (Version 1.0.0) [Data set]. Zenodo. http://doi.org/10.5281/zenodo.4743746*

More specifically, a preprocessed version which combines around 6 million samples from this dataset into one singular csv file.

## 1. Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

## 2. Load dataset

In [2]:
dataset = Path("data/iot23.parquet")
df_raw = pd.read_parquet(dataset)

In [3]:
df = df_raw.copy()
df.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,label
0,1.536227e+09,CeqqKl3hyLQmO8LK98,192.168.100.111,17576.0,78.1.220.212,8081.0,tcp,-,3e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
1,1.536227e+09,C2oHQWo1EFGH8D9x7,192.168.100.111,17576.0,152.84.7.111,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
2,1.536227e+09,CJLVjs4BByG04mczXc,192.168.100.111,17576.0,173.36.41.67,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
3,1.536227e+09,C0z4uS9AWHDH2s4S7,192.168.100.111,17576.0,87.13.21.104,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan
4,1.536227e+09,CxbNVk3liFNUIlqSPi,192.168.100.111,17576.0,99.110.163.140,8081.0,tcp,-,2e-06,0,...,S0,-,-,0.0,S,2.0,80.0,0.0,0.0,PartOfAHorizontalPortScan


## 3. EDA

Rename columns

In [4]:
df.rename(columns={
    "ts": "timestamp",
    "id.orig_h": "source_addr",
    "id.orig_p": "source_port",
    "id.resp_h": "dest_addr",
    "id.resp_p": "dest_port",
    "proto": "protocol",
    "orig_bytes": "payload_bytes_sent_from_source",
    "resp_bytes": "payload_bytes_sent_from_dest",
    "conn_state": "connection_state",
    "orig_pkts": "packets_sent_from_source",
    "resp_pkts": "packets_sent_from_dest",
    "orig_ip_bytes": "ip_bytes_sent_from_source",
    "resp_ip_bytes": "ip_bytes_sent_from_dest"
}, inplace=True)

## 4. Feature Engineering

Convert labels into normal (0) or attack (1)

In [5]:
df['is_attack'] = np.where(df['label'] == 'Benign', 0, 1).astype(bool)
df = df.drop(columns=['label'])

In [6]:
df['is_attack'].value_counts()

is_attack
True     5357811
False     688812
Name: count, dtype: int64

In [7]:
print(df.dtypes)

timestamp                         float64
uid                                   str
source_addr                           str
source_port                       float64
dest_addr                             str
dest_port                         float64
protocol                              str
service                               str
duration                              str
payload_bytes_sent_from_source        str
payload_bytes_sent_from_dest          str
connection_state                      str
local_orig                            str
local_resp                            str
missed_bytes                      float64
history                               str
packets_sent_from_source          float64
ip_bytes_sent_from_source         float64
packets_sent_from_dest            float64
ip_bytes_sent_from_dest           float64
is_attack                            bool
dtype: object


In [8]:
# List the columns that should be numbers but are currently 'str'
numeric_cols = [
    'duration',
    'payload_bytes_sent_from_source',
    'payload_bytes_sent_from_dest'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill the new NaNs with 0
df[numeric_cols] = df[numeric_cols].fillna(0)

In [9]:
# List of columns that should be whole numbers
int_cols = [
    'source_port', 'dest_port', 'missed_bytes',
    'packets_sent_from_source', 'packets_sent_from_dest',
    'payload_bytes_sent_from_source', 'payload_bytes_sent_from_dest',
    'ip_bytes_sent_from_source', 'ip_bytes_sent_from_dest'
]

df[int_cols] = df[int_cols].astype('Int64')

In [10]:
# List of columns that should be categorical
cat_cols = ['protocol', 'service', 'connection_state', 'history']

for col in cat_cols:
    df[col] = df[col].astype('category')

In [11]:
# Find columns with only 1 unique value
single_value_cols = [col for col in df.columns if df[col].nunique() <= 1]

print(f"Columns to drop (only 1 unique value): {len(single_value_cols)}")
print(single_value_cols)

Columns to drop (only 1 unique value): 2
['local_orig', 'local_resp']


Drop columns

In [12]:
df.drop(columns=["uid", "local_orig", "local_resp"], inplace=True)
df.head()

,timestamp,source_addr,source_port,dest_addr,dest_port,protocol,service,duration,payload_bytes_sent_from_source,payload_bytes_sent_from_dest,connection_state,missed_bytes,history,packets_sent_from_source,ip_bytes_sent_from_source,packets_sent_from_dest,ip_bytes_sent_from_dest,is_attack
0,1.536227e+09,192.168.100.111,17576,78.1.220.212,8081,tcp,-,0.000003,0,0,S0,0,S,2,80,0,0,True
1,1.536227e+09,192.168.100.111,17576,152.84.7.111,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True
2,1.536227e+09,192.168.100.111,17576,173.36.41.67,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True
3,1.536227e+09,192.168.100.111,17576,87.13.21.104,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True
4,1.536227e+09,192.168.100.111,17576,99.110.163.140,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True


Create Inter-Arrival-Time (IAT) feature

In [13]:
df["inter_arrival_time"] = df["timestamp"].diff().fillna(0)
df.drop(columns=["timestamp"], inplace=True)
df.head()

,source_addr,source_port,dest_addr,dest_port,protocol,service,duration,payload_bytes_sent_from_source,payload_bytes_sent_from_dest,connection_state,missed_bytes,history,packets_sent_from_source,ip_bytes_sent_from_source,packets_sent_from_dest,ip_bytes_sent_from_dest,is_attack,inter_arrival_time
0,192.168.100.111,17576,78.1.220.212,8081,tcp,-,0.000003,0,0,S0,0,S,2,80,0,0,True,0.000000
1,192.168.100.111,17576,152.84.7.111,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True,0.000005
2,192.168.100.111,17576,173.36.41.67,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True,0.000004
3,192.168.100.111,17576,87.13.21.104,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True,0.000012
4,192.168.100.111,17576,99.110.163.140,8081,tcp,-,0.000002,0,0,S0,0,S,2,80,0,0,True,0.000004


Create avg packet size feature

In [14]:
df['avg_packet_size_source'] = (df['ip_bytes_sent_from_source'] / df['packets_sent_from_source']).fillna(0)
df['avg_packet_size_dest'] = (df['ip_bytes_sent_from_dest'] / df['packets_sent_from_dest']).fillna(0)

Create byte to packet ratio feature

In [15]:
df['byte_packet_ratio'] = (df['ip_bytes_sent_from_source'] / df['packets_sent_from_source']).fillna(0)

Create throughput feature

In [16]:
df['throughput'] = (df['ip_bytes_sent_from_source'] / df['duration']).fillna(0)

In [17]:
df.drop(columns=['source_addr', 'dest_addr', 'history'], inplace=True)

One-hot encode catagorical features

In [18]:
# Select the categorical columns
categorical_cols = ['protocol', 'service', 'connection_state']

# This will transform them into multiple 0/1 columns
df = pd.get_dummies(df, columns=categorical_cols)

In [19]:
# Convert everything to standard floats
df = df.astype(float)

In [20]:
df.head()

,source_port,dest_port,duration,payload_bytes_sent_from_source,payload_bytes_sent_from_dest,missed_bytes,packets_sent_from_source,ip_bytes_sent_from_source,packets_sent_from_dest,ip_bytes_sent_from_dest,...,connection_state_RSTOS0,connection_state_RSTR,connection_state_RSTRH,connection_state_S0,connection_state_S1,connection_state_S2,connection_state_S3,connection_state_SF,connection_state_SH,connection_state_SHR
0,17576.0,8081.0,0.000003,0.0,0.0,0.0,2.0,80.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,17576.0,8081.0,0.000002,0.0,0.0,0.0,2.0,80.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,17576.0,8081.0,0.000002,0.0,0.0,0.0,2.0,80.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,17576.0,8081.0,0.000002,0.0,0.0,0.0,2.0,80.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,17576.0,8081.0,0.000002,0.0,0.0,0.0,2.0,80.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
# 1. Replace any remaining hidden inf values with 0
df = df.replace([np.inf, -np.inf], 0)

# 2. Fill any remaining NaNs with 0
df = df.fillna(0)

Scale values

In [22]:
scaler = RobustScaler() # RobustScaler is great for data with outliers like yours
X = scaler.fit_transform(df.drop(columns=['is_attack']))
y = df['is_attack']

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Built-in feature importance
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

perm = permutation_importance(
    rf, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

print(perm_importance.head(20))